# Pruning vs XNOR-Net — ESC-50 project, in one notebook

**Readable, local Windows/Jupyter version of our reproduction work.** This notebook consolidates the custom Python workflow into ordered, explained cells instead of long terminal commands.

**Flow:** environment → ESC-50 folds → ACDNet/XNOR model setup → optional training → five-fold validation → physical channel pruning → optional fine-tuning → real CPU INT8 post-training quantization → comparisons and export.

> **Scope:** Our completed runs use the **ESC-10 class subset of ESC-50** (200 training epochs, five folds), then a lightweight **5%-per-layer L2 channel-pruning** experiment. This is **not** the original paper's 2,000-epoch, 10–50-class, iterative Taylor-pruning reproduction. The XMini inference here uses PyTorch's binary-weight simulation, not packed 1-bit CPU kernels.

**Repository dependencies:** This is a self-contained notebook for our *custom experiment code*, but still imports the original authors' `audio.nets.models`, `audio.resources.utils`, `audio.resources.esc_subsets`, and `common.binop_lib`. Keep it inside your existing `pruning-vs-xnor` repository. You will also need the previously prepared `wav20.npz` and saved checkpoints for the no-retraining path.

## 0. Open the notebook in the correct Python environment

1. Save this `.ipynb` file in `C:\AI_Projects\pruning-vs-xnor`.
2. In VS Code, choose **Select Kernel → Python Environments → `venv_gpu`** (Python 3.10.11, PyTorch 2.9.1+cu128).
3. If notebook support is missing, run once in PowerShell:

```powershell
.\venv_gpu\Scripts\python.exe -m pip install ipykernel jupyter pandas matplotlib nbformat
```

Then select **Run All**. By default, the notebook **reuses existing checkpoints/results without long training or redoing quantization**. To recompute any stage, turn on the corresponding flag in Section 1.

## 1. Imports and configuration

Edit paths **once here**. Everything below uses these settings, rather than repeating long command-line arguments. GPU is used for full-precision/XMini computation; quantized INT8 inference is CPU-only.

In [ ]:
import copy
import csv
import json
import os
import random
import sys
import time
import warnings
from collections import defaultdict
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from IPython.display import display

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# --- Local Windows paths (edit these if your project is elsewhere) ---
PROJECT_ROOT = Path(r'C:\AI_Projects\pruning-vs-xnor')
DATA_ROOT = Path(r'C:\AI_Projects\datasets\esc50')
EXPERIMENT_ROOT = PROJECT_ROOT / 'experiments' / 'esc50'
NOTEBOOK_OUTPUT = EXPERIMENT_ROOT / 'notebook_outputs'

# --- What to RUN? (False = read existing results and avoid unnecessary work) ---
RUN_NEW_TRAINING = False            # True = train Mini/XMini from scratch
RUN_LITE_COMPRESSION = False       # True = run 5-fold pruning + INT8 again

# --- Optional training settings ---
TRAIN_MODELS = ['mini', 'xmini']
TRAIN_CLASSES = [10]                # May also use [10, 20, 30, 40, 50]
TRAIN_FOLDS = [1, 2, 3, 4, 5]
TRAIN_EPOCHS = 200
TRAIN_BATCH_SIZE = 64
TRAIN_EVAL_EVERY = 10
TRAIN_SAVE_EVERY = 10
TRAIN_RESUME = False               # Only resume identical training settings

# --- Lightweight compression settings (our final no-fine-tune experiment) ---
LITE_FOLDS = [1, 2, 3, 4, 5]
PRUNE_FRACTION = 0.05               # 5% of each eligible TFEB Conv output channels
FINETUNE_EPOCHS = 0                # 0 => direct pruning -> INT8
FINETUNE_STEPS = 1
FINETUNE_BATCH_SIZE = 16
CALIBRATION_SAMPLES = 12
VAL_PER_CLASS = 4                   # 4 x 10 = 40 validation clips per fold
EVAL_CROPS = 3
EVAL_BATCH = 2
SEED = 42

# The compression results folder we already created in PowerShell:
EXISTING_LITE_CSV = EXPERIMENT_ROOT / 'lite_prune5_no_finetune' / 'comparison.csv'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Selected device:', DEVICE)


### Verify the repository and dataset paths

The `wav20.npz` is outside the repository in your setup. The original authors' source modules remain in the repository; our notebook does not duplicate their architecture definitions.

In [ ]:
if not PROJECT_ROOT.is_dir():
    # Helpful when opening notebook from the project directory on another computer.
    if (Path.cwd() / 'audio').is_dir() and (Path.cwd() / 'common').is_dir():
        PROJECT_ROOT = Path.cwd().resolve()
        EXPERIMENT_ROOT = PROJECT_ROOT / 'experiments' / 'esc50'
        NOTEBOOK_OUTPUT = EXPERIMENT_ROOT / 'notebook_outputs'
    else:
        raise FileNotFoundError(f'Project directory does not exist: {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for relative in ['audio/nets/models.py', 'audio/resources/utils.py',
                 'audio/resources/esc_subsets.py', 'common/binop_lib.py']:
    if not (PROJECT_ROOT / relative).is_file():
        raise FileNotFoundError(f'Missing original repository module: {relative}')

DATA_FILE = DATA_ROOT / 'wav20.npz'
if not DATA_FILE.is_file():
    raise FileNotFoundError(
        f'Missing {DATA_FILE}. Use your prepared ESC-50 wav20.npz, or update DATA_ROOT.')

NOTEBOOK_OUTPUT.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Dataset:', DATA_FILE)
print('Notebook output:', NOTEBOOK_OUTPUT)


In [ ]:
# Original repository modules — these stay as imports, not copied code.
from audio.nets import models
from audio.resources import esc_subsets
from audio.resources import utils as U
from common.binop_lib import BinOp

INPUT_LENGTH = 30225        # 1.51125 seconds at 20 kHz
SAMPLE_RATE = 20000
N_CROPS = 10                 # Used for the 200-epoch full-fold evaluation

print('Original ACDNet/XNOR model code imported successfully.')


## 2. Dataset loading and fold-safe splitting

ESC-50 contains **2,000 recordings in five official folds**. In each ESC-10 run we train on four folds (320 clips) and validate on the held-out fold (80 clips). The class indices come from the paper's subset definition implemented in the repository, not a random subset. During training we mix recordings with *Between-Class learning* augmentation.

In [ ]:
def get_subset(n_classes):
    if n_classes == 50:
        return list(range(50))
    return list(getattr(esc_subsets, 'esc{}'.format(n_classes)))

def load_folds(path):
    if not os.path.isfile(path):
        raise IOError('Missing {}. Use the wav20.npz built yesterday.'.format(path))
    with np.load(path, allow_pickle=True) as archive:
        folds = {}
        for fold in range(1, 6):
            obj = archive['fold{}'.format(fold)].item()
            folds[fold] = list(zip(obj['sounds'], [int(x) for x in obj['labels']]))
    print('Loaded ESC-50: {}'.format(', '.join('fold{}={}'.format(k, len(v)) for k, v in sorted(folds.items()))))
    return folds

def select_fold(fold_data, test_fold, classes):
    mapping = {original: new for new, original in enumerate(classes)}
    train = []
    test = []
    for fold, records in fold_data.items():
        destination = test if fold == test_fold else train
        for sound, label in records:
            if label in mapping:
                destination.append((sound, mapping[label]))
    expected_train = 32 * len(classes)
    expected_test = 8 * len(classes)
    if len(train) != expected_train or len(test) != expected_test:
        raise ValueError('Unexpected ESC-50 split: training={} (expected {}), test={} (expected {})'.format(
            len(train), expected_train, len(test), expected_test))
    return train, test

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [ ]:
set_seed(SEED)
all_folds = load_folds(str(DATA_FILE))
class_ids = get_subset(10)
train_fold1, val_fold1 = select_fold(all_folds, test_fold=1, classes=class_ids)
print('ESC-10 original class indices:', class_ids)
print('Fold 1 training clips:', len(train_fold1))
print('Fold 1 validation clips:', len(val_fold1))


### Optional data preview

Visualize one raw audio recording and the distribution of the labels in Fold 1. Audio values in `wav20.npz` are read directly; no spectrogram conversion is used in this project.

In [ ]:
audio, audio_label = train_fold1[0]
plt.figure(figsize=(10, 2.3))
plt.plot(np.arange(len(audio)) / SAMPLE_RATE, audio, linewidth=0.7)
plt.title(f'ESC-10 example: remapped class {audio_label}')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (archive scale)')
plt.tight_layout()
plt.show()

counts = pd.Series([label for _, label in train_fold1]).value_counts().sort_index()
display(counts.rename('Training clips per class').to_frame().T)


## 3. Model architecture and checkpoints

* **Mini-ACDNet:** full-precision convolution model using the authors' Mini channel configuration.
* **XMini-ACDNet:** corresponding binary-weight/activation model, using `BinOp` from the original repository.

**Critical reproduction safeguard:** Architecture checkpoints in `audio/models/*.pt` supply only the **channel configuration**. During a new fold's training, we **do not load pretrained weights** from those files because that can leak information from an incompatible fold. To evaluate already trained fold-specific models, Section 6 loads our own checkpoint `weight` dictionaries.

In [ ]:
def load_local_checkpoint(path, map_location='cpu'):
    # Only load files this project created locally. Never unpickle unknown files.
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:  # older PyTorch
        return torch.load(path, map_location=map_location)


In [ ]:
def get_config(model_name):
    if model_name in ['acdnet', 'xacdnet']:
        return None
    fname = 'audio/models/{}_acdnet.pt'.format(model_name[1:] if model_name == 'xmini' else model_name)
    if not os.path.isfile(fname):
        raise IOError('Missing channel-config checkpoint: {}'.format(fname))
    # Important: only take the layer/channel counts, NOT these checkpoint weights.
    payload = load_local_checkpoint(fname, map_location='cpu')
    if not isinstance(payload, dict) or 'config' not in payload:
        raise ValueError('Checkpoint does not contain config: {}'.format(fname))
    return payload['config']

def create_model(model_name, n_classes, device):
    config = get_config(model_name)
    is_binary = model_name.startswith('x')
    factory = models.GetBINACDNetModel if is_binary else models.GetACDNetModel
    model = factory(INPUT_LENGTH, n_classes, SAMPLE_RATE, config).to(device)
    binop = BinOp(model) if is_binary else None
    return model, binop, config


In [ ]:
# Instantiate only when training is enabled (loading all model weights is Section 6).
if RUN_NEW_TRAINING:
    demo_model, demo_binop, demo_config = create_model('mini', 10, DEVICE)
    print('Mini parameter count:', sum(p.numel() for p in demo_model.parameters()))
    del demo_model, demo_binop
else:
    print('Training is off — no randomly initialized model created.')


## 4. Between-Class augmentation and full-fold validation

This is a notebook conversion of the code in `esc50_repro_runner_v2.py`. It generates augmented training batches **on demand**, avoiding the original repository's pre-generated per-epoch audio files. Validation averages predictions across **10 fixed crops** per recording, which is different from the later lightweight test (3 crops and 40 recordings/fold).

In [ ]:
class BCLearningBatches(object):
    """Reproduce audio/resources/train_generator.py mixing logic on demand."""
    def __init__(self, records, n_classes):
        self.records = records
        self.n_classes = n_classes
        self.labels = np.eye(n_classes, dtype=np.float32)
        self.scale = U.random_scale(1.25)
        self.pad = U.padding(INPUT_LENGTH // 2)
        self.crop = U.random_crop(INPUT_LENGTH)
        self.norm = U.normalize(32768.0)
        self.gain = U.random_gain(6)

    def augment(self, audio):
        return self.norm(self.crop(self.pad(self.scale(audio))))

    def batch(self, batch_size, device):
        x = np.empty((batch_size, 1, 1, INPUT_LENGTH), dtype=np.float32)
        y = np.empty((batch_size, self.n_classes), dtype=np.float32)
        for i in range(batch_size):
            a, label_a = random.choice(self.records)
            b, label_b = random.choice(self.records)
            while label_a == label_b:
                b, label_b = random.choice(self.records)
            a = self.augment(a)
            b = self.augment(b)
            r = random.random()
            mixed = U.mix(a, b, np.array(r), SAMPLE_RATE).astype(np.float32)
            x[i, 0, 0, :] = self.gain(mixed).astype(np.float32)
            y[i, :] = self.labels[label_a] * r + self.labels[label_b] * (1 - r)
        return torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)

def validation_crops(audio):
    # Same normalization/crop order as original val_generator.py.
    padded = U.padding(INPUT_LENGTH // 2)(audio)
    normalized = U.normalize(32768.0)(padded)
    return U.multi_crop(INPUT_LENGTH, N_CROPS)(normalized).astype(np.float32)

def evaluate(model, binop, records, n_classes, device, sample_batch):
    model.eval()
    if binop is not None:
        binop.binarization()
    correct = 0
    total_loss = 0.0
    try:
        with torch.no_grad():
            for idx in range(0, len(records), sample_batch):
                group = records[idx: idx + sample_batch]
                stacked = np.stack([validation_crops(audio) for audio, _ in group], axis=0)
                tensor = torch.from_numpy(stacked.reshape(-1, 1, 1, INPUT_LENGTH)).to(device)
                output = model(tensor).reshape(len(group), N_CROPS, n_classes).mean(dim=1)
                targets = torch.tensor([label for _, label in group], dtype=torch.long, device=device)
                correct += int((output.argmax(dim=1) == targets).sum().item())
                selected = output.gather(1, targets.view(-1, 1)).clamp(min=1e-8)
                total_loss += float((-selected.log()).sum().item())
    finally:
        if binop is not None:
            binop.restore()
        model.train()
    return (100.0 * correct / len(records), total_loss / len(records))

def learning_rate(epoch, max_epochs):
    # Mirrors the published repository's __get_lr, including 10-epoch warmup.
    decay = sum(epoch > max_epochs * fraction for fraction in [0.3, 0.6, 0.9])
    if epoch <= 10:
        decay = 1
    return 0.1 * (0.1 ** decay)


## 5. Optional new training (200 epochs, with resumable checkpoints)

We have **already trained** all five ESC-10 folds for Mini and XMini, so `RUN_NEW_TRAINING=False` by default. Turning it on starts independent runs using the original repository's optimizer approach: SGD + stepwise learning-rate schedule for Mini, Adam + cosine scheduler for XMini. Separate per-epoch metadata and restart checkpoints are saved.

**Do not change the total epoch count when resuming.** The learning-rate schedule is a function of the original planned training length. The best accuracy is chosen using the held-out fold, which can create optimistic selection bias.

In [ ]:
def append_csv(path, row, fields):
    exists = os.path.isfile(path)
    with open(path, 'a', newline='') as fp:
        writer = csv.DictWriter(fp, fieldnames=fields)
        if not exists:
            writer.writeheader()
        writer.writerow(row)

def save_atomic(payload, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + '.tmp'
    torch.save(payload, tmp)
    os.replace(tmp, path)

def capture_rng_state():
    state = {'python': random.getstate(),
             'numpy': np.random.get_state(),
             'torch': torch.get_rng_state()}
    if torch.cuda.is_available():
        state['cuda'] = torch.cuda.get_rng_state_all()
    return state

def restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'].cpu())
    if torch.cuda.is_available() and 'cuda' in state:
        torch.cuda.set_rng_state_all(state['cuda'])

def reconcile_history_for_resume(csv_path, target_model, n_classes, fold, last_epoch, fields):
    # A crash may leave history rows newer than the most recent state snapshot.
    if not os.path.isfile(csv_path):
        return
    with open(csv_path, 'r', newline='') as fp:
        rows = list(csv.DictReader(fp))
    kept = []
    removed = 0
    for row in rows:
        match = (row['model'] == target_model and int(row['classes']) == n_classes
                 and int(row['fold']) == fold)
        if match and int(row['epoch']) > last_epoch:
            removed += 1
        else:
            kept.append(row)
    if removed:
        tmp = csv_path + '.tmp'
        with open(tmp, 'w', newline='') as fp:
            writer = csv.DictWriter(fp, fieldnames=fields)
            writer.writeheader()
            writer.writerows(kept)
        os.replace(tmp, csv_path)
        print('Reconciled history: dropped {} rows after saved epoch {}'.format(removed, last_epoch))


In [ ]:
def train_one(args, folds, n_classes, test_fold, device, csv_path):
    set_seed(args.seed)
    selected_classes = get_subset(n_classes)
    train, validation = select_fold(folds, test_fold, selected_classes)
    model, binop, config = create_model(args.model, n_classes, device)
    print('\n=== {} / {} classes / fold {} ==='.format(args.model, n_classes, test_fold), flush=True)
    print('Train={}  Validation={}  Device={}  Parameters={:,}'.format(
        len(train), len(validation), device, sum(p.numel() for p in model.parameters())), flush=True)
    is_binary = binop is not None
    if is_binary:
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=0.1, weight_decay=5e-4,
                                    momentum=0.9, nesterov=True)
        scheduler = None
    loss_fn = torch.nn.KLDivLoss(reduction='batchmean')
    source = BCLearningBatches(train, n_classes)
    steps = args.steps_per_epoch or (len(train) // args.batch_size)
    if steps < 1:
        raise ValueError('steps-per-epoch must be >= 1')
    best_acc = -1.0
    best_path = os.path.join(args.output, 'checkpoints', '{}_c{}_fold{}.pt'.format(args.model, n_classes, test_fold))
    state_path = os.path.join(args.output, 'checkpoints', '{}_c{}_fold{}_trainstate.pt'.format(args.model, n_classes, test_fold))
    fields = ['model', 'classes', 'fold', 'epoch', 'train_accuracy', 'train_loss',
              'validation_accuracy', 'validation_loss', 'lr', 'seconds', 'steps', 'batch_size', 'seed']
    run_metadata = {'model': args.model, 'classes': n_classes, 'fold': test_fold,
                    'seed': args.seed, 'epochs': args.epochs, 'batch_size': args.batch_size,
                    'steps': steps, 'eval_every': args.eval_every}
    start_epoch = 1
    if args.resume:
        if not os.path.isfile(state_path):
            has_history = False
            if os.path.isfile(csv_path):
                with open(csv_path, 'r', newline='') as fp:
                    has_history = any(row['model'] == args.model
                                      and int(row['classes']) == n_classes
                                      and int(row['fold']) == test_fold
                                      for row in csv.DictReader(fp))
            if has_history:
                raise IOError('History exists but no trainstate checkpoint was found: {}'.format(state_path))
            print('No previous state for this fold; starting it from epoch 1.', flush=True)
        else:
            saved = load_local_checkpoint(state_path)
            if saved.get('metadata') != run_metadata:
                raise ValueError('Resume settings do not match saved run. Saved: {} | Current: {}'.format(
                    saved.get('metadata'), run_metadata))
            model.load_state_dict(saved['model_state'])
            optimizer.load_state_dict(saved['optimizer_state'])
            if scheduler is not None:
                scheduler.load_state_dict(saved['scheduler_state'])
            best_acc = float(saved['best_acc'])
            start_epoch = int(saved['epoch']) + 1
            restore_rng_state(saved['rng_state'])
            reconcile_history_for_resume(csv_path, args.model, n_classes, test_fold,
                                         start_epoch - 1, fields)
            print('Resuming {} class {} fold {} from epoch {} (best {:.2f}%)'.format(
                args.model, n_classes, test_fold, start_epoch, best_acc), flush=True)
    elif os.path.isfile(csv_path):
        with open(csv_path, 'r', newline='') as fp:
            for row in csv.DictReader(fp):
                if (row['model'] == args.model and int(row['classes']) == n_classes
                        and int(row['fold']) == test_fold):
                    raise ValueError('History exists for this run. Use a NEW --output directory '
                                     'or --resume (if a v2 trainstate checkpoint exists).')
    for epoch in range(start_epoch, args.epochs + 1):
        start = time.time()
        model.train()
        if not is_binary:
            lr = learning_rate(epoch, args.epochs)
            optimizer.param_groups[0]['lr'] = lr
        else:
            lr = optimizer.param_groups[0]['lr']
        correct_accum = 0.0
        loss_accum = 0.0
        for _ in range(steps):
            x, y = source.batch(args.batch_size, device)
            optimizer.zero_grad()
            if binop is not None:
                binop.binarization()
            outputs = model(x)
            loss = loss_fn(outputs.clamp(min=1e-8).log(), y)
            correct_accum += float((outputs.argmax(1) == y.argmax(1)).float().mean().item())
            loss_accum += float(loss.item())
            loss.backward()
            if binop is not None:
                binop.restore()
                binop.updateBinaryGradWeight()
            optimizer.step()
        if scheduler is not None:
            scheduler.step()
        val_acc, val_loss = float('nan'), float('nan')
        if epoch % args.eval_every == 0 or epoch == args.epochs:
            val_acc, val_loss = evaluate(model, binop, validation, n_classes, device,
                                         args.eval_sample_batch)
            if val_acc > best_acc:
                best_acc = val_acc
                torch.save({'weight': model.state_dict(), 'config': list(config) if config is not None else None,
                            'meta': {'model': args.model, 'classes': n_classes, 'fold': test_fold,
                                     'epoch': epoch, 'accuracy': val_acc, 'seed': args.seed}}, best_path)
        seconds = time.time() - start
        row = dict(model=args.model, classes=n_classes, fold=test_fold, epoch=epoch,
                   train_accuracy=round(correct_accum / steps * 100, 4),
                   train_loss=round(loss_accum / steps, 6),
                   validation_accuracy=round(val_acc, 4), validation_loss=round(val_loss, 6),
                   lr=lr, seconds=round(seconds, 2), steps=steps,
                   batch_size=args.batch_size, seed=args.seed)
        append_csv(csv_path, row, fields)
        if epoch % args.save_every == 0 or epoch == args.epochs:
            save_atomic({'epoch': epoch, 'metadata': run_metadata,
                         'model_state': model.state_dict(),
                         'optimizer_state': optimizer.state_dict(),
                         'scheduler_state': scheduler.state_dict() if scheduler is not None else None,
                         'best_acc': best_acc, 'rng_state': capture_rng_state()}, state_path)
        print('epoch {}/{} | train {:.2f}% loss {:.4f} | val {:.2f}% loss {:.4f} | {:.1f}s'.format(
            epoch, args.epochs, row['train_accuracy'], row['train_loss'],
            val_acc, val_loss, seconds), flush=True)
    return best_acc, best_path


In [ ]:
if RUN_NEW_TRAINING:
    for model_name in TRAIN_MODELS:
        # Separate folders by model/classes/epoch count; never overwrite old experiments.
        for n_classes in TRAIN_CLASSES:
            out_dir = EXPERIMENT_ROOT / f'notebook_train_{model_name}_c{n_classes}_e{TRAIN_EPOCHS}'
            (out_dir / 'checkpoints').mkdir(parents=True, exist_ok=True)
            config = SimpleNamespace(
                model=model_name, output=str(out_dir), seed=SEED,
                epochs=TRAIN_EPOCHS, batch_size=TRAIN_BATCH_SIZE,
                steps_per_epoch=0, eval_every=TRAIN_EVAL_EVERY,
                eval_sample_batch=2, save_every=TRAIN_SAVE_EVERY,
                resume=TRAIN_RESUME,
            )
            for held_out in TRAIN_FOLDS:
                best_acc, best_checkpoint = train_one(
                    config, all_folds, n_classes, held_out, DEVICE,
                    str(out_dir / 'history.csv'))
                print(f'{model_name}, ESC-{n_classes}, fold {held_out}: '
                      f'best={best_acc:.2f}% -> {best_checkpoint}')
else:
    print('Skipped 200-epoch training: reusing saved local checkpoints and histories.')


## 6. Full ESC-10 five-fold evaluation — existing training histories

The historic 200-epoch experiment evaluated **80 held-out recordings per fold**, using **10 crops per recording**. We show each model's **best validation checkpoint** per fold. If a history file is missing, the notebook labels the fallback values as **previously transcribed terminal output**, rather than pretending the CSV was loaded.

In [ ]:
# Our previously recorded 200-epoch results, used only as an explicitly labelled fallback.
REPORTED_200EPOCH = {
    'Mini-ACDNet': [87.50, 92.50, 88.75, 96.25, 91.25],
    'XMini-ACDNet': [63.75, 72.50, 66.25, 70.00, 73.75],
}

HISTORY_LOCATIONS = {
    'Mini-ACDNet': [
        EXPERIMENT_ROOT / 'mini10_200epochs' / 'history.csv',
        EXPERIMENT_ROOT / 'mini10_cv200_folds2to5' / 'history.csv',
    ],
    'XMini-ACDNet': [
        EXPERIMENT_ROOT / 'xmini10_200epochs' / 'history.csv',
        EXPERIMENT_ROOT / 'xmini10_cv200_folds2to5' / 'history.csv',
    ],
}

full_rows = []
for model_label, paths in HISTORY_LOCATIONS.items():
    pieces = []
    for path in paths:
        if path.is_file():
            piece = pd.read_csv(path)
            piece['source_path'] = str(path.relative_to(PROJECT_ROOT))
            pieces.append(piece)
    frame = pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame()
    for fold in range(1, 6):
        values = pd.DataFrame()
        if not frame.empty:
            values = frame.loc[pd.to_numeric(frame['fold'], errors='coerce') == fold].copy()
            values['epoch'] = pd.to_numeric(values['epoch'], errors='coerce')
            values['validation_accuracy'] = pd.to_numeric(
                values['validation_accuracy'], errors='coerce')
        complete = (not values.empty and (values['epoch'] == 200).any())
        valid = values.dropna(subset=['validation_accuracy']) if complete else pd.DataFrame()
        if not valid.empty:
            best = valid.sort_values('validation_accuracy', ascending=False).iloc[0]
            acc = float(best['validation_accuracy'])
            best_epoch = int(best['epoch'])
            source_name = 'history.csv'
        else:
            acc = REPORTED_200EPOCH[model_label][fold - 1]
            best_epoch = None
            source_name = 'previous terminal output (not rechecked here)'
        full_rows.append({'model': model_label, 'fold': fold, 'best_accuracy_%': acc,
                          'best_epoch': best_epoch, 'source': source_name})

full_cv = pd.DataFrame(full_rows)
display(full_cv)
display(full_cv.groupby('model', as_index=False)['best_accuracy_%'].mean()
        .rename(columns={'best_accuracy_%':'five_fold_mean_%'}))


In [ ]:
pivot = full_cv.pivot(index='fold', columns='model', values='best_accuracy_%')
ax = pivot.plot(kind='bar', figsize=(9, 4.0), rot=0)
ax.set(title='ESC-10 — best validation accuracy by held-out fold (200 epochs)',
       xlabel='Held-out fold', ylabel='Accuracy (%)', ylim=(0, 100))
ax.legend(title='')
plt.tight_layout()
plt.show()


## 7. Loading our trained models for compression comparison

This is the same fold-specific checkpoint mapping used in our original lightweight script. We **never** substitute the architecture-only checkpoint from `audio/models` for trained weights here. All models are evaluated on exactly the same balanced validation recordings within each fold.

In [ ]:
def load_trusted_local(path):
    # Only use locally created experiment checkpoints: torch.load may unpickle.
    if not os.path.isfile(path):
        raise FileNotFoundError('Missing checkpoint: ' + path)
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')

def locate_checkpoint(model_name, fold):
    if fold == 1:
        folder = 'mini10_200epochs' if model_name == 'mini' else 'xmini10_200epochs'
    else:
        folder = 'mini10_cv200_folds2to5' if model_name == 'mini' else 'xmini10_cv200_folds2to5'
    return os.path.join('experiments', 'esc50', folder, 'checkpoints',
                        '{}_c10_fold{}.pt'.format(model_name, fold))

def load_checkpoint_model(model_name, fold, device):
    path = locate_checkpoint(model_name, fold)
    payload = load_trusted_local(path)
    if 'weight' not in payload or 'config' not in payload:
        raise ValueError('Unexpected checkpoint format: ' + path)
    meta = payload.get('meta', {})
    if meta and (int(meta.get('classes', 10)) != 10 or int(meta.get('fold', fold)) != fold):
        raise ValueError('Checkpoint metadata mismatch in {}: {}'.format(path, meta))
    config = list(payload['config'])
    factory = models.GetBINACDNetModel if model_name == 'xmini' else models.GetACDNetModel
    net = factory(INPUT_LENGTH, 10, SAMPLE_RATE, config).to(device)
    net.load_state_dict(payload['weight'], strict=True)
    net.eval()
    binop = BinOp(net) if model_name == 'xmini' else None
    return net, binop, path

def balanced_samples(records, per_class, seed):
    by_label = defaultdict(list)
    for audio, label in records:
        by_label[label].append((audio, int(label)))
    rng = random.Random(seed)
    selected = []
    for label in sorted(by_label):
        items = list(by_label[label])
        rng.shuffle(items)
        if per_class and len(items) < per_class:
            raise ValueError('Not enough validation clips in class {}'.format(label))
        selected.extend(items[:per_class] if per_class else items)
    rng.shuffle(selected)
    return selected

def crops(audio, n_crops):
    padded = U.padding(INPUT_LENGTH // 2)(audio)
    normalized = U.normalize(32768.0)(padded)
    if n_crops == 1:
        start = (len(normalized) - INPUT_LENGTH) // 2
        arr = normalized[start:start + INPUT_LENGTH][np.newaxis, :]
    else:
        arr = U.multi_crop(INPUT_LENGTH, n_crops)(normalized)
    return np.asarray(arr, dtype=np.float32)

def precompute_eval_batches(records, n_crops, recs_per_batch):
    result = []
    for i in range(0, len(records), recs_per_batch):
        group = records[i:i + recs_per_batch]
        batch = np.stack([crops(sound, n_crops) for sound, _ in group], axis=0)
        x = torch.from_numpy(batch.reshape(-1, 1, 1, INPUT_LENGTH).copy())
        y = torch.tensor([label for _, label in group], dtype=torch.long)
        result.append((x, y))
    return result

def score_model(model, batches, device, n_crops, binop=None):
    model.eval()
    correct = 0
    n = 0
    loss_total = 0.0
    if binop is not None:
        binop.binarization()
    try:
        with torch.inference_mode():
            for xb, yb in batches:
                xb = xb.to(device)
                yb = yb.to(device)
                out = model(xb).reshape(len(yb), n_crops, 10).mean(1)
                if out.shape != (len(yb), 10) or not torch.isfinite(out).all():
                    raise AssertionError('Invalid prediction tensor')
                correct += int((out.argmax(1) == yb).sum().item())
                loss_total += float(-out.gather(1, yb[:, None]).clamp(min=1e-8).log().sum().item())
                n += len(yb)
    finally:
        if binop is not None:
            binop.restore()
    return round(correct * 100.0 / n, 2), round(loss_total / n, 5)


## 8. Physical structured channel pruning (L2 ranking)

We rank output channels by the L2 norm of their convolution weights, remove the lowest-ranked channels, and **rewire the following Conv/BatchNorm/Linear layer**. This changes tensor dimensions and genuinely reduces parameter count. For the 5% setting, the network loses **102,875 of 1,046,396** FP32 parameters (about **9.8%**).

This is one-shot **per-layer L2 pruning**, **not** the paper's iterative global Taylor/hybrid criterion. A large per-layer fraction can disproportionately shrink the model. The optional fine-tune is brief and is not the paper's repeated retraining.

In [ ]:
def clone_conv(conv, new_in, new_out):
    if conv.groups != 1:
        raise ValueError('Grouped convolutions not supported')
    return nn.Conv2d(new_in, new_out, conv.kernel_size, stride=conv.stride,
                     padding=conv.padding, dilation=conv.dilation,
                     groups=conv.groups, bias=(conv.bias is not None),
                     padding_mode=conv.padding_mode)

def prune_tfeb_l2(model, fraction):
    """One-shot L2 ranking of each TFEB Conv; physical output/input rewiring."""
    if not 0 < fraction < 0.8:
        raise ValueError('--prune-fraction should be >0 and <0.8')
    net = copy.deepcopy(model).cpu().eval()
    conv_positions = [i for i, m in enumerate(net.tfeb) if isinstance(m, nn.Conv2d)]
    if len(conv_positions) != 10:
        raise RuntimeError('Expected 10 TFEB Conv2d layers, found {}'.format(len(conv_positions)))
    li = next((i for i, m in enumerate(net.tfeb) if isinstance(m, nn.Linear)), None)
    if li is None:
        raise RuntimeError('Missing TFEB final Linear layer')
    changes = []
    for j, ci in enumerate(conv_positions):
        layer = net.tfeb[ci]
        if ci + 1 >= len(net.tfeb) or not isinstance(net.tfeb[ci + 1], nn.BatchNorm2d):
            raise RuntimeError('Expected Conv-BN at index {}'.format(ci))
        bn = net.tfeb[ci + 1]
        count = layer.out_channels
        remove_n = int(round(count * fraction))
        if count >= 4:
            remove_n = max(1, remove_n)
        remove_n = min(remove_n, count - 2)
        if remove_n <= 0:
            continue
        keep_n = count - remove_n
        scores = layer.weight.detach().square().flatten(1).sum(1)
        keep = torch.topk(scores, keep_n, largest=True).indices.sort().values
        next_idx = conv_positions[j + 1] if j + 1 < len(conv_positions) else li
        next_module = net.tfeb[next_idx]
        if isinstance(next_module, nn.Conv2d):
            if next_module.in_channels != count:
                raise RuntimeError('Next conv shape mismatch: {} vs {}'.format(next_module.in_channels, count))
            new_next = clone_conv(next_module, keep_n, next_module.out_channels)
        elif isinstance(next_module, nn.Linear):
            if next_module.in_features != count:
                raise RuntimeError('Next linear input mismatch: {} vs {}'.format(next_module.in_features, count))
            new_next = nn.Linear(keep_n, next_module.out_features,
                                 bias=(next_module.bias is not None))
        else:
            raise RuntimeError('Unsupported downstream layer')
        new_layer = clone_conv(layer, layer.in_channels, keep_n)
        new_bn = nn.BatchNorm2d(keep_n, eps=bn.eps, momentum=bn.momentum,
                                affine=bn.affine, track_running_stats=bn.track_running_stats)
        with torch.no_grad():
            new_layer.weight.copy_(layer.weight[keep])
            if layer.bias is not None:
                new_layer.bias.copy_(layer.bias[keep])
            if bn.affine:
                new_bn.weight.copy_(bn.weight[keep])
                new_bn.bias.copy_(bn.bias[keep])
            if bn.track_running_stats:
                new_bn.running_mean.copy_(bn.running_mean[keep])
                new_bn.running_var.copy_(bn.running_var[keep])
                new_bn.num_batches_tracked.copy_(bn.num_batches_tracked)
            if isinstance(next_module, nn.Conv2d):
                new_next.weight.copy_(next_module.weight[:, keep])
                if next_module.bias is not None:
                    new_next.bias.copy_(next_module.bias)
            else:
                new_next.weight.copy_(next_module.weight[:, keep])
                if next_module.bias is not None:
                    new_next.bias.copy_(next_module.bias)
        net.tfeb[ci] = new_layer
        net.tfeb[ci + 1] = new_bn
        net.tfeb[next_idx] = new_next
        # Config indexes 2-11 correspond to TFEB Conv3-Conv12.
        net.ch_config = list(net.ch_config)
        net.ch_config[2 + j] = keep_n
        changes.append({'conv': j + 3, 'before': count, 'after': keep_n})
    if not changes:
        raise RuntimeError('No channels removed')
    return net, changes

def finetune_pruned(model, training, device, epochs, steps, batch_size):
    if epochs <= 0:
        return model
    model.to(device).train()
    batches = BCLearningBatches(training, 10)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)
    kl = nn.KLDivLoss(reduction='batchmean')
    for epoch in range(epochs):
        losses = []
        for _ in range(steps):
            xx, yy = batches.batch(batch_size, device)
            optimizer.zero_grad(set_to_none=True)
            prob = model(xx)
            loss = kl(prob.clamp(min=1e-8).log(), yy)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.item()))
        print('  Fine-tune {}/{}: train loss {:.4f}'.format(epoch + 1, epochs, sum(losses) / len(losses)), flush=True)
    return model.cpu().eval()


## 9. CPU INT8 post-training quantization

After pruning, the PyTorch eager-mode flow is:

`Conv + BatchNorm + ReLU fusion` → `prepare observers` → `calibrate on training-fold clips` → `convert to INT8` → `CPU inference`.

We pick a CPU backend supported by this particular PyTorch build (including **oneDNN**, as on our Windows GPU environment). An INT8 checkpoint is a *real quantized inference graph*, not merely an FP32 tensor converted to 8-bit storage. Supported operator conversion and file size depend on the backend/runtime.

In [ ]:
def fuse_conv_bn_relu(seq):
    fused = 0
    for i in range(len(seq) - 2):
        if (isinstance(seq[i], nn.Conv2d) and isinstance(seq[i + 1], nn.BatchNorm2d)
                and isinstance(seq[i + 2], nn.ReLU)):
            torch.ao.quantization.fuse_modules(seq, [str(i), str(i + 1), str(i + 2)], inplace=True)
            fused += 1
    return fused

def quantize_int8(pruned, calibration, calib_samples):
    candidate = models.GetACDNetModel(INPUT_LENGTH, 10, SAMPLE_RATE,
                                       list(pruned.ch_config), quantize=True).cpu().eval()
    candidate.load_state_dict(pruned.state_dict(), strict=True)
    fused = fuse_conv_bn_relu(candidate.sfeb) + fuse_conv_bn_relu(candidate.tfeb)
    if fused == 0:
        raise RuntimeError('Could not fuse Conv-BN-ReLU layers')
    supported = list(torch.backends.quantized.supported_engines)
    engine = next((e for e in ('x86', 'fbgemm', 'onednn', 'qnnpack') if e in supported), None)
    if engine is None:
        raise RuntimeError('No supported CPU INT8 backend: {}'.format(supported))
    torch.backends.quantized.engine = engine
    candidate.qconfig = torch.ao.quantization.get_default_qconfig(engine)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=DeprecationWarning)
        torch.ao.quantization.prepare(candidate, inplace=True)
    samples = balanced_samples(calibration, 0, seed=42)
    with torch.inference_mode():
        for audio, _ in samples[:calib_samples]:
            one = torch.from_numpy(crops(audio, 1).reshape(1, 1, 1, INPUT_LENGTH).copy())
            candidate(one)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=DeprecationWarning)
        torch.ao.quantization.convert(candidate, inplace=True)
    int8_count = sum(isinstance(m, torch.ao.nn.quantized.Conv2d) for m in candidate.modules())
    if int8_count < 1:
        raise AssertionError('No quantized convolutions were created')
    return candidate.eval(), engine, int8_count, fused

def count_params(model):
    return int(sum(p.numel() for p in model.parameters()))


## 10. Optional five-fold lightweight reproduction

Set `RUN_LITE_COMPRESSION=True` in Section 1 to recompute the entire pipeline. It loads the **existing 200-epoch trained checkpoints**, performs 5% physical pruning, optionally fine-tunes, calibrates INT8 with 12 clips, exports/reloads the quantized model, and saves a new CSV.

Default settings match our **no-fine-tuning** run: five folds, 40 clips/fold, 3 crops, prune fraction 5%, and 12 calibration samples. We write to a **new notebook-only output folder**, never overwriting the previous terminal experiments.

In [ ]:
# Reusable wrapper around the functions above. No argparse or command-line calls.
def run_lightweight_comparison():
    set_seed(SEED)
    torch.set_num_threads(min(4, torch.get_num_threads()))
    destination = EXPERIMENT_ROOT / 'notebook_prune5_no_finetune'
    destination.mkdir(parents=True, exist_ok=True)
    all_results = []

    for fold in LITE_FOLDS:
        print(f'\n===== ESC-10 / fold {fold} =====', flush=True)
        set_seed(SEED + fold)
        training, full_validation = select_fold(all_folds, fold, get_subset(10))
        validation = balanced_samples(full_validation, VAL_PER_CLASS, SEED + fold)
        batches = precompute_eval_batches(validation, EVAL_CROPS, EVAL_BATCH)

        mini, _, mini_ckpt = load_checkpoint_model('mini', fold, DEVICE)
        xmini, binop, xmini_ckpt = load_checkpoint_model('xmini', fold, DEVICE)
        mini_acc, mini_loss = score_model(mini, batches, DEVICE, EVAL_CROPS)
        xmini_acc, xmini_loss = score_model(xmini, batches, DEVICE, EVAL_CROPS, binop)
        original_n, binary_n = count_params(mini), count_params(xmini)

        pruned, changes = prune_tfeb_l2(mini.cpu(), PRUNE_FRACTION)
        del mini, xmini, binop
        pruned_n = count_params(pruned)
        pruned_acc, pruned_loss = score_model(pruned, batches, torch.device('cpu'), EVAL_CROPS)
        tuned = finetune_pruned(pruned, training, DEVICE, FINETUNE_EPOCHS,
                                 FINETUNE_STEPS, FINETUNE_BATCH_SIZE)
        tuned_acc, tuned_loss = score_model(tuned, batches, torch.device('cpu'), EVAL_CROPS)
        quantized, backend, n_int8, fused = quantize_int8(tuned, training, CALIBRATION_SAMPLES)
        int8_acc, int8_loss = score_model(quantized, batches, torch.device('cpu'), EVAL_CROPS)

        fold_dir = destination / f'fold{fold}'
        fold_dir.mkdir(parents=True, exist_ok=True)
        pruned_file = fold_dir / 'pruned_mini_fp32.pt'
        torch.save({'weight': tuned.state_dict(), 'config': list(tuned.ch_config),
                    'finetune_epochs': FINETUNE_EPOCHS}, str(pruned_file))
        int8_file = fold_dir / 'pruned_mini_int8_traced.pt'
        test_input = batches[0][0][:1].cpu()
        with torch.inference_mode():
            trace = torch.jit.trace(quantized, test_input, check_trace=False)
            torch.jit.save(trace, str(int8_file))
            restored = torch.jit.load(str(int8_file), map_location='cpu').eval()
            if not torch.allclose(quantized(test_input), restored(test_input), atol=1e-5, rtol=1e-5):
                raise AssertionError('INT8 export/reload mismatch')

        variants = [
            ('Mini FP32 (existing)', mini_acc, mini_loss, original_n, mini_ckpt),
            ('XMini PyTorch (existing)', xmini_acc, xmini_loss, binary_n, xmini_ckpt),
            ('Mini L2-channel-pruned', pruned_acc, pruned_loss, pruned_n, ''),
            ('Mini pruned + short fine-tune' if FINETUNE_EPOCHS else 'Mini pruned (no fine-tune)',
             tuned_acc, tuned_loss, pruned_n, str(pruned_file)),
            ('Mini pruned + fine-tune + INT8' if FINETUNE_EPOCHS else 'Mini pruned + INT8 (no fine-tune)',
             int8_acc, int8_loss, pruned_n, str(int8_file)),
        ]
        for name, acc, loss, npar, artifact in variants:
            all_results.append({
                'fold': fold, 'variant': name, 'validation_clips': len(validation),
                'eval_crops': EVAL_CROPS, 'accuracy_percent': acc, 'mean_nll': loss,
                'fp32_model_parameters': npar,
                'artifact_mb': round(os.path.getsize(artifact)/1048576, 3) if artifact else '',
                'artifact': artifact,
            })
            print(f'  {name:36s} {acc:6.2f}%')

        (fold_dir / 'compression_info.json').write_text(json.dumps({
            'fold': fold, 'pruning': changes, 'backend': backend,
            'int8_conv_count': n_int8, 'fused_blocks': fused,
            'finetune_epochs': FINETUNE_EPOCHS, 'calibration_samples': CALIBRATION_SAMPLES,
            'validation_clips': len(validation), 'eval_crops': EVAL_CROPS,
        }, indent=2), encoding='utf-8')

    csv_file = destination / 'comparison.csv'
    pd.DataFrame(all_results).to_csv(csv_file, index=False)
    print('Saved:', csv_file)
    return pd.DataFrame(all_results)

if RUN_LITE_COMPRESSION:
    notebook_lite = run_lightweight_comparison()
else:
    notebook_lite = None
    print('Skipped compression re-run: use existing comparison.csv in Section 11.')


## 11. Compare the previous five-fold compression experiments

By default this section reads the already saved CSVs:

- **20%** channel pruning + 1 very short fine-tuning epoch (`lite_fivefold`)
- **5%** channel pruning + 3 short fine-tuning epochs (`lite_prune5_fivefold`)
- **5%** channel pruning **without** fine-tuning (`lite_prune5_no_finetune`)

The CSV summaries use **40 validation recordings/fold and 3 crops**. They are *not* directly comparable to the full 200-epoch validation accuracy in Section 6 (80 recordings/fold, 10 crops).

In [ ]:
COMPRESSION_RUNS = {
    '20% prune + short tune': EXPERIMENT_ROOT / 'lite_fivefold' / 'comparison.csv',
    '5% prune + short tune': EXPERIMENT_ROOT / 'lite_prune5_fivefold' / 'comparison.csv',
    '5% prune + no tune': EXISTING_LITE_CSV,
}
previous = []
for run_name, path in COMPRESSION_RUNS.items():
    if path.is_file():
        df = pd.read_csv(path)
        expected = {'fold','variant','accuracy_percent'}
        if not expected.issubset(df.columns):
            raise ValueError(f'{path} does not contain required columns: {expected}')
        df['run'] = run_name
        # The original script retains a 'fine-tune' row label even when epochs=0.
        # Rename only this no-fine-tune run for clarity; do not change saved CSVs.
        if run_name == '5% prune + no tune':
            df['variant'] = df['variant'].replace({
                'Mini pruned + short fine-tune': 'Mini pruned (no fine-tune)',
                'Mini pruned + fine-tune + INT8': 'Mini pruned + INT8 (no fine-tune)',
            })
        previous.append(df)
    else:
        print('Not found (skipping):', path)

if notebook_lite is not None:
    rerun = notebook_lite.copy()
    rerun['run'] = 'notebook recomputation'
    previous.append(rerun)

if previous:
    compression_df = pd.concat(previous, ignore_index=True)
    compression_means = (compression_df.groupby(['run','variant'], as_index=False)
                         .agg(mean_accuracy_percent=('accuracy_percent','mean'),
                              folds=('fold','nunique')))
    display(compression_means.sort_values(['run','mean_accuracy_percent'],
                                           ascending=[True,False]))
else:
    compression_df = pd.DataFrame()
    compression_means = pd.DataFrame()
    print('No compression CSVs were found. Turn RUN_LITE_COMPRESSION=True, or copy old CSVs here.')


In [ ]:
if not compression_df.empty:
    # Main five-fold no-fine-tune results (the cleanest low-load comparison).
    filtered = compression_df.loc[compression_df['run'] == '5% prune + no tune']
    if not filtered.empty:
        summary = filtered.groupby('variant')['accuracy_percent'].mean().sort_values(ascending=False)
        display(summary.rename('Five-fold mean accuracy (%)').to_frame())
        plt.figure(figsize=(10, 4))
        ax = summary.sort_values().plot(kind='barh')
        ax.set(title='ESC-10 — 5% pruning and direct INT8 (5-fold mean)',
               xlabel='Accuracy (%)', xlim=(0, 100), ylabel='')
        plt.tight_layout()
        plt.show()


## 12. Compression size: parameters versus actual files

**Pruning** physically changed model dimensions from **1,046,396 to 943,521 parameters** (a **9.8%** reduction). The `prune-fraction=0.05` flag refers to each eligible layer, not directly to the global parameter percentage.

**INT8** requires a separate measurement of saved artifact size. Counting FP32 model parameters is *not* the same thing as measuring a mixed-precision quantized model. Likewise, XMini's `state_dict` size is **not** the 1-bit packed inference size claimed theoretically in the paper.

In [ ]:
parameters_before = 1_046_396
parameters_after = 943_521
print(f'Physically removed parameters: {parameters_before - parameters_after:,}')
print(f'Parameter reduction: {(1 - parameters_after / parameters_before)*100:.2f}%')

if not compression_df.empty:
    artifacts = compression_df[['run','fold','variant','artifact_mb']].copy()
    artifacts['artifact_mb'] = pd.to_numeric(artifacts['artifact_mb'], errors='coerce')
    display(artifacts.dropna(subset=['artifact_mb']).head(15))
    print('Saved sizes above include serialization overhead and are not directly MCU RAM use.')


## 13. Comparison with the 2022 IEEE Access paper

**Paper:** M. Mohaimenuzzaman, C. Bergmeir, and B. Meyer, *Pruning vs XNOR-Net: A Comprehensive Study of Deep Learning for Audio Classification on Edge-Devices* (IEEE Access, 2022), Table 6.

We compare **ESC-10 full-fold training results** with the paper's ESC-10 Mini/XMini values, not with the lightweight 40-clip compression results. This is a **contextual benchmark only** because we used 200 instead of 2,000 epochs and our checkpoints were selected on the evaluated folds.

In [ ]:
paper_esc10 = {'Mini-ACDNet': 96.75, 'XMini-ACDNet': 82.25}
our_esc10 = full_cv.groupby('model')['best_accuracy_%'].mean().to_dict()
paper_compare = pd.DataFrame([
    {'model': model, 'our_200_epoch_best_five_fold_%': our_esc10.get(model),
     'paper_2000_epoch_five_fold_%': published,
     'difference_pp': our_esc10.get(model, float('nan'))-published}
    for model, published in paper_esc10.items()
])
display(paper_compare)

paper_compare.set_index('model')[['our_200_epoch_best_five_fold_%',
                                  'paper_2000_epoch_five_fold_%']].plot(
    kind='bar', rot=0, figsize=(8,4), ylim=(0,100),
    title='ESC-10 — abbreviated reproduction vs paper', ylabel='Accuracy (%)')
plt.tight_layout()
plt.show()


## 14. Results summary, export, and scientific limitations

### Findings

- The existing **200-epoch five-fold ESC-10 runs** achieved **91.25%** for Mini-ACDNet and **69.25%** for XMini-ACDNet.
- In the reduced **40-clip-per-fold, three-crop** comparison, FP32 Mini achieved **84.50%**, 5%-pruned Mini **76.00%**, pruned→INT8 without tuning **72.50%**, and XMini **58.00%**.
- The more aggressive 20%-per-layer L2 pruning dropped mean pruned accuracy to **21.00%**, demonstrating that pruning strength matters.
- The short fine-tuning protocol did not consistently help: with the 5% setting, mean pruned FP32 accuracy changed from **76.00% without fine-tuning** to **69.00% after short fine-tuning**.

### Limits of this notebook / reproduction

1. ESC-10 subset only in completed tests; the full ESC-20/30/40/50 matrix was not trained.
2. 200 training epochs instead of the paper's 2,000.
3. L2 one-shot TFEB pruning instead of the paper's iterative Taylor-based/hybrid channel pruning; it is **not Micro/QMicro-ACDNet**.
4. XMini uses PyTorch binary-weight operations, not optimized hardware 1-bit XNOR+popcount; don't infer bit-level speedups from normal timing.
5. The FP32/INT8/XMini models are **not memory-matched**, so these accuracies do not prove which is better under an equal RAM budget.
6. Checkpoints were selected by best accuracy on the same fold subsequently reported, so accuracy may be optimistically biased.
7. oneDNN may warn about CPU instruction support; PTQ quantized accuracy can vary with backend/calibration.

**Practical conclusion:** The code paths for raw-audio ACDNet/XNOR training, held-out evaluation, physical channel pruning, calibration, INT8 inference, and export all function on the local setup. Our completed results are an **abbreviated functional reproduction**, not an exact replication of the paper's final performance tables.

In [ ]:
# Save small, human-readable summaries next to the notebook. Existing experiments remain untouched.
full_cv.to_csv(NOTEBOOK_OUTPUT / 'full_200epoch_esc10_fivefold.csv', index=False)
paper_compare.to_csv(NOTEBOOK_OUTPUT / 'paper_comparison_esc10.csv', index=False)
if not compression_means.empty:
    compression_means.to_csv(NOTEBOOK_OUTPUT / 'lite_compression_fivefold_means.csv', index=False)
print('Summary CSVs written to:', NOTEBOOK_OUTPUT)


---

### Troubleshooting

**`ModuleNotFoundError: audio`** — Place the notebook in the original GitHub repository and set `PROJECT_ROOT` correctly. The notebook intentionally imports the authors' model architecture and binary ops.

**`FileNotFoundError: wav20.npz`** — Your archive is at `C:\AI_Projects\datasets\esc50\wav20.npz`, *beside* the repository; update `DATA_ROOT`, not the model path. This notebook does not silently download or recreate the dataset.

**No trained checkpoints** — Recopy your existing `experiments/esc50` folders, or set `RUN_NEW_TRAINING=True` to train a model with a fresh output directory. Compression runs require trained **ESC-10 fold-specific** weights.

**Only oneDNN quantization backend available** — The quantization code explicitly supports `onednn`; quantized Conv inference runs on **CPU**.

**Interrupting training** — Use the notebook's separate `notebook_train_...` output folder and set `TRAIN_RESUME=True` only with unchanged settings. Never resume a 500-epoch checkpoint as a 200-epoch experiment.

**Research reference:** [Original authors' GitHub repository](https://github.com/mohaimenz/pruning-vs-xnor) · [IEEE Access paper DOI](https://doi.org/10.1109/ACCESS.2022.3140807).